# 110 — Anatomía: instrucciones, herramientas, estado y salida

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**Anatomía de todo agente LLM — cuatro piezas:**

- **Instrucciones:** la política versionable — objetivo verificable, restricciones,
  procedimiento, formato de salida. Lo no escrito queda a criterio del modelo.
- **Herramientas:** contratos (nombre, descripción, JSON Schema). El modelo **emite
  intenciones** de llamada; el runtime valida y ejecuta — esa separación permite
  interponer permisos y aprobaciones.
- **Estado:** tres capas con vidas distintas — contexto (ventana, volátil y cara),
  estado de la tarea (plan/progreso/presupuesto, estructurado), memoria persistente
  (entre runs).
- **Salida estructurada:** esquema verificable mecánicamente. En este programa:
  `{kind, seed, result, evidence, limitations}` — evidencia y límites obligatorios.


### 🔩 El esqueleto en el laboratorio

| Pieza | En `run_lab("agent")` |
|---|---|
| Instrucciones | objetivo "verificar estado y sumar 7 + 5" + condición de parada |
| Herramientas | `status()` (lectura), `sum(left, right)` (pura) |
| Estado | `trace` + condiciones verificadas |
| Salida | contrato JSON con `evidence` y `limitations` |

Todo framework de agentes (Claude SDK, OpenAI Agents, LangGraph) es una implementación
opinada de este mismo esqueleto.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Disección del contrato.** Ejecuta `run_lab("agent", seed=110)` y escribe
un validador mecánico de la salida: claves de nivel superior, `evidence` no vacía, y cada
elemento de `trace` con la forma `{action: {tool, args}, observation}`. ¿Cuántos `assert`
necesitas para validar el contrato completo?

**Ejercicio 2 — Instrucciones operativas.** Convierte esta instrucción vaga en una
política de cuatro bloques (rol/objetivo, restricciones, procedimiento, formato):
"Eres un asistente que ayuda con los reportes de gastos". Incluye al menos una condición
de parada y una acción que requiera aprobación.

**Ejercicio 3 — Clasifica el estado.** Para un agente que revisa pull requests, asigna
cada dato a su capa (contexto / estado de la tarea / memoria persistente) y justifica:
(a) el diff del PR actual; (b) la lista de archivos ya revisados en este run; (c) las
convenciones de estilo del equipo; (d) el presupuesto de pasos restante; (e) los
comentarios que el agente ya escribió; (f) qué patrones de bug aparecieron en PRs de
meses anteriores.

**Ejercicio 4 — Diseña la cuarta pieza.** El laboratorio devuelve `final: {healthy, sum}`.
Propón el esquema JSON de salida para el agente revisor de PRs del ejercicio 3: campos
obligatorios, tipos, y dónde viven `evidence` y `limitations`. Escríbelo como esquema
comentado en la celda de código.


In [ ]:
# TODO: ejecuta run_lab("agent", seed=110)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: validador mecánico del contrato
result = run_lab("agent", seed=110)

def valida_contrato(r):
    # completa: claves de nivel superior
    # completa: evidence no vacía
    # completa: forma de cada elemento de trace
    return True

print(valida_contrato(result))


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 4: esquema de salida del agente revisor de PRs
esquema_salida = {
    # "campo": ("tipo", "obligatorio u opcional", "para qué sirve"),
}


## Reflexión

1. La separación "el modelo emite intenciones, el runtime ejecuta" parece un detalle de
   implementación. Nombra dos controles concretos (de clases 116-117) que serían
   imposibles si el modelo ejecutara herramientas directamente.
2. El contrato del laboratorio obliga a `evidence` y `limitations` en la salida. ¿Qué
   verificación mecánica permite cada campo, y qué se pierde si el resultado final del
   agente fuera prosa libre?
3. ¿Por qué "meter todo el historial al contexto" y "no registrar estado estructurado de
   la tarea" son el mismo error visto desde dos capas distintas del estado? ¿Qué operación
   (reanudar, auditar, cobrar) rompe cada uno?
